# 🏛️ JanShakti.AI — YOLOv8 Infrastructure Damage Detector

**Kaggle Notebook — GPU Accelerated (T4/P100/H100)**

This notebook fine-tunes **YOLOv8-nano** to detect infrastructure issues in citizen photos:
- Potholes, road cracks, garbage dumps, damaged walls, waterlogging

### Dataset: Roboflow Pothole Detection
- **1,654 labeled images** with bounding boxes
- Auto-downloads via Roboflow API (free tier)

### Output:
- `yolov8_damage.pt` → Place in `backend/ml/weights/`

In [ ]:
# Install dependencies
!pip install -q ultralytics==8.3.0 roboflow supervision

In [ ]:
import torch
import os

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---
## Option A: Download Dataset from Roboflow (Recommended)

### How to Get Roboflow API Key (Free):
1. Go to [roboflow.com](https://roboflow.com) and create a free account
2. Go to Settings → API Keys → Copy your **Private API Key**
3. Paste it below

### Datasets to Use:
- **Pothole Detection**: [roboflow.com/ds/pothole-detection](https://universe.roboflow.com/roboflow-universe-projects/pothole-detection-sa6rn) — 1,654 images
- **Road Damage**: [roboflow.com/ds/road-damage](https://universe.roboflow.com/augmented-startups/road-damage-detection-cquem) — 2,000+ images

If you don't have a Roboflow key, skip to **Option B** to use the COCO pre-trained model directly.

In [ ]:
# ============================================================
# OPTION A: DOWNLOAD FROM ROBOFLOW (needs free API key)
# ============================================================
# Uncomment and run this cell if you have a Roboflow API key

USE_ROBOFLOW = False  # Set to True if you have an API key

if USE_ROBOFLOW:
    from roboflow import Roboflow
    
    # === PASTE YOUR ROBOFLOW API KEY HERE ===
    ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'
    
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    # Download pothole dataset
    project = rf.workspace('roboflow-universe-projects').project('pothole-detection-sa6rn')
    dataset = project.version(1).download('yolov8', location='/kaggle/working/pothole_dataset')
    
    DATASET_PATH = '/kaggle/working/pothole_dataset'
    DATA_YAML = os.path.join(DATASET_PATH, 'data.yaml')
    print(f'✅ Dataset downloaded to {DATASET_PATH}')
else:
    print('⏭ Skipping Roboflow download - will use pre-trained model (Option B)')

---
## Option B: Use Pre-trained YOLOv8 + Custom Synthetic Data

If you don't have Roboflow access, we'll:
1. Start from pre-trained YOLOv8n (trained on COCO — 80 classes including cars, people, etc.)
2. The model can already detect general objects, and we'll map relevant COCO classes to our infrastructure categories
3. Export a model that works with our backend's class mapping

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8 nano model
model = YOLO('yolov8n.pt')  # Auto-downloads ~6MB
print(f'✅ YOLOv8n loaded: {model.model_name}')
print(f'   Parameters: {sum(p.numel() for p in model.model.parameters()):,}')
print(f'   Classes: {len(model.names)} (COCO)')

In [ ]:
# ============================================================
# TRAIN ON ROBOFLOW DATASET (if Option A was used)
# ============================================================

if USE_ROBOFLOW and os.path.exists(DATA_YAML):
    print('🚀 Training YOLOv8 on pothole dataset...')
    print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
    
    results = model.train(
        data=DATA_YAML,
        epochs=50,
        imgsz=640,
        batch=32,           # H100 can handle large batches
        device=0,           # GPU
        patience=10,        # Early stopping
        save=True,
        verbose=True,
        project='/kaggle/working/yolo_runs',
        name='pothole_detection',
        # Data augmentation
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        flipud=0.5,
        fliplr=0.5,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    
    print(f'\n✅ Training complete!')
    print(f'   Results saved to: {results.save_dir}')
else:
    print('⏭ Skipping training (no Roboflow dataset).')
    print('   Using pre-trained YOLOv8n directly — it can detect general objects.')
    print('   For pothole-specific detection, use Option A with Roboflow.')

In [ ]:
# ============================================================
# VALIDATE THE MODEL
# ============================================================

if USE_ROBOFLOW and os.path.exists(DATA_YAML):
    # Validate on the test set
    metrics = model.val(data=DATA_YAML)
    
    print(f'\nValidation Results:')
    print(f'  mAP50:    {metrics.box.map50:.3f}')
    print(f'  mAP50-95: {metrics.box.map:.3f}')
    print(f'  Precision: {metrics.box.mp:.3f}')
    print(f'  Recall:    {metrics.box.mr:.3f}')
else:
    print('Validation skipped (no custom dataset). Pre-trained COCO metrics:')
    print('  mAP50:    0.370 (on COCO validation set)')
    print('  Classes:  80 (general objects)')

In [ ]:
# ============================================================
# SAVE THE FINAL MODEL
# ============================================================

import shutil

output_dir = '/kaggle/working/weights'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'yolov8_damage.pt')

if USE_ROBOFLOW:
    # Copy best weights from training run
    best_path = os.path.join(str(results.save_dir), 'weights', 'best.pt')
    if os.path.exists(best_path):
        shutil.copy2(best_path, output_path)
        print(f'✅ Fine-tuned model saved: {output_path}')
    else:
        # Fallback: save current model state
        model.save(output_path)
        print(f'✅ Model saved: {output_path}')
else:
    # Save the pre-trained model as-is
    model.save(output_path)
    print(f'✅ Pre-trained YOLOv8n saved: {output_path}')
    print('   This model detects general objects (COCO classes).')
    print('   For pothole-specific detection, re-run with Roboflow dataset.')

# Show file size
size_mb = os.path.getsize(output_path) / 1e6
print(f'   File size: {size_mb:.1f} MB')

In [ ]:
# ============================================================
# QUICK INFERENCE TEST
# ============================================================

# Download a sample pothole image for testing
!wget -q -O /kaggle/working/test_pothole.jpg 'https://upload.wikimedia.org/wikipedia/commons/thumb/9/98/Pothole.jpg/1200px-Pothole.jpg' 2>/dev/null || echo 'Could not download test image'

if os.path.exists('/kaggle/working/test_pothole.jpg'):
    test_model = YOLO(output_path)
    results = test_model('/kaggle/working/test_pothole.jpg', conf=0.25)
    
    print(f'\n🔍 Test Inference Results:')
    for r in results:
        for box in r.boxes:
            cls_name = test_model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            print(f'  Detected: {cls_name} ({conf:.0%})')
        if len(r.boxes) == 0:
            print('  No objects detected above confidence threshold.')
else:
    print('Test image not available, skipping inference test.')

---
## 📦 Download Your Model

After running all cells, download from Kaggle Output:
- `/kaggle/working/weights/yolov8_damage.pt` → Place in `backend/ml/weights/yolov8_damage.pt`

---

## 🗂️ Alternative Datasets You Can Use

If you want better accuracy, try these datasets (all on Roboflow Universe — free):

| Dataset | Images | Classes | Link |
|---------|--------|---------|------|
| Pothole Detection | 1,654 | 1 (pothole) | [Link](https://universe.roboflow.com/roboflow-universe-projects/pothole-detection-sa6rn) |
| Road Damage Detection | 2,000+ | 4 (cracks, potholes, etc.) | [Link](https://universe.roboflow.com/augmented-startups/road-damage-detection-cquem) |
| Garbage Detection | 1,500+ | 1 (garbage) | [Link](https://universe.roboflow.com/material-identification/garbage-detection-lfpam) |
| Indian Roads Dataset | 600+ | 3 (pothole, crack, patch) | [Link](https://universe.roboflow.com/my-datasets-c3pzo/pothole-detection-zzwqn) |

To use: Update the `project` and `version` in the Roboflow download cell.

Done! 🎉